In [1]:
import torch
from transformers import AutoProcessor, AutoModel
from PIL import Image
import json

In [2]:
# Configuration
MODEL_NAME = "ds4sd/SmolDocling-256M-preview"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESUME_PATH = "other_resume_samples/4f1m9ybl7d1a1.png"  # Change this to your resume

print(f"Loading model on {DEVICE}...")

Loading model on cpu...


In [4]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    trust_remote_code=True,
).to(DEVICE)

if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("Model loaded!\n")

Model loaded!



In [6]:
print(f"Loading resume: {RESUME_PATH}")
image = Image.open(RESUME_PATH)

# Resize if needed (for 4GB GPU)
if max(image.size) > 1024:
    image.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
    print(f"  Resized to: {image.size}")

Loading resume: other_resume_samples/4f1m9ybl7d1a1.png


In [7]:
# Create prompt
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {
                "type": "text",
                "text": """Extract all information from this resume and format as JSON:
                {
                  "personal_info": {
                    "name": "",
                    "email": "",
                    "phone": "",
                    "location": ""
                  },
                  "work_experience": [
                    {
                      "title": "",
                      "company": "",
                      "dates": "",
                      "responsibilities": []
                    }
                  ],
                  "education": [
                    {
                      "degree": "",
                      "institution": "",
                      "graduation_date": ""
                    }
                  ],
                  "skills": []
                }

                Extract all available information."""
            }
        ],
    },
]

In [8]:
# Process
print("Processing...")
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt").to(DEVICE)

# Generate
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        pad_token_id=processor.tokenizer.pad_token_id,
    )

# Decode (only new tokens)
prompt_length = inputs['input_ids'].shape[1]
generated_texts = processor.batch_decode(
    generated_ids[:, prompt_length:],
    skip_special_tokens=True,
)

Processing...


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


AttributeError: 'Idefics3Model' object has no attribute 'generate'

In [ ]:
result = generated_texts[0].strip()

# Display result
print("\n" + "="*60)
print("PARSED RESUME:")
print("="*60)

In [ ]:
try:
    import re
    cleaned = re.sub(r'```json\s*|\s*```', '', result)
    parsed = json.loads(cleaned)
    print(json.dumps(parsed, indent=2))

    # Save to file
    with open("parsed_resume.json", "w") as f:
        json.dump(parsed, f, indent=2)
    print("\n💾 Saved to: parsed_resume.json")

except json.JSONDecodeError:
    print(result)
    print("\n⚠️ Output is not valid JSON")

    # Save raw text
    with open("parsed_resume.txt", "w") as f:
        f.write(result)
    print("💾 Saved raw text to: parsed_resume.txt")

# Cleanup
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("\nDone!")